# 🎯 Module 03: Regression

> **Stage:** Machine Learning  
> **Level:** Beginner → Intermediate  
> **Module:** 03 — Regression  
> **Date:** 05 January 2026  
> **Previous:** 02 — Data Preprocessing  
> **Next:** 04 — Classification

---

**Regression** is the family of supervised-learning models that predict **continuous numerical values**.

In this notebook we'll cover:

- What regression is
- **Linear Regression** (single & multiple features)
- How models **learn** (minimizing MSE)
- **Polynomial Regression** for non-linear patterns
- **Ridge** (L2), **Lasso** (L1), **ElasticNet** (L1 + L2)
- Evaluation metrics: **MAE, MSE, RMSE, R²**
- Common problems: overfitting, multicollinearity

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what regression is (and how it differs from classification)
- Fit a **Linear Regression** model and interpret `coef_` and `intercept_`
- Explain what **MSE** means and why it's used as a loss
- Use **Polynomial Regression** to capture non-linear patterns
- Explain **regularization** and why **Ridge**, **Lasso**, and **ElasticNet** exist
- Use **Lasso** for implicit feature selection
- Compute and interpret **MAE, MSE, RMSE, R²**
- Diagnose **overfitting** and **multicollinearity**

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
print('✅ Setup ready.')

---
## 🧠 1. What is Regression?

**Regression** is a supervised ML technique used to predict a **numerical value**.

### Examples

- House price → `15,500,000`
- Salary → `180,000`
- Temperature → `32.5°C`
- Sales → `250,000`
- Delivery time → `42.7 minutes`

### Basic idea

```text
Features (X)
    ↓
Regression Model
    ↓
Numerical Prediction (y)
```

```text
Area + Bedrooms + Location
          ↓
     ML Model
          ↓
     House Price
```

---
## 📐 2. Linear Regression

Linear Regression assumes the relationship between features and target can be approximated by a **linear equation**.

### One feature

$$y = mx + b$$

| Symbol | Meaning |
| --- | --- |
| `y` | prediction |
| `x` | feature |
| `m` | slope / weight |
| `b` | intercept |

### Multiple features

$$y = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

```text
House Price =
    5000 × Area
+ 100000 × Bedrooms
+ 200000 × Bathrooms
+ b
```

The model **learns** the weights from training data.

---
## 🎯 3. How Does Linear Regression Learn?

```text
Weights + Bias
     ↓
Prediction
     ↓
Compare with actual value
     ↓
Calculate error
     ↓
Optimize parameters
```

### The objective: minimize MSE

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

| Symbol | Meaning |
| --- | --- |
| $y_i$ | actual value |
| $\hat{y}_i$ | predicted value |
| $n$ | number of observations |

The model tries to make predictions as **close as possible** to the actual values.

In [ ]:
# Create a simple synthetic dataset for regression
np.random.seed(42)
n = 200

area     = np.random.randint(500, 3000, n)
bedrooms = np.random.randint(1, 6, n)
bathrooms = np.random.randint(1, 4, n)
age      = np.random.randint(0, 30, n)

# True relationship + noise
price = (5000 * area
         + 100000 * bedrooms
         + 200000 * bathrooms
         - 50000 * age
         + 500000
         + np.random.randn(n) * 250000)

df = pd.DataFrame({
    'area': area,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age': age,
    'price': price,
})

print('Dataset shape:', df.shape)
print(df.head())
print('\nStatistical summary:')
print(df.describe().round(0))

---
## 💻 4. Linear Regression with Scikit-Learn

In [ ]:
# Prepare X and y
X = df[['area', 'bedrooms', 'bathrooms', 'age']]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print('Learned coefficients:')
for feat, coef in zip(X.columns, model.coef_):
    print(f'  {feat:10s} → {coef:>15,.2f}')
print(f'\nIntercept: {model.intercept_:,.2f}')

# Evaluate
mae  = mean_absolute_error(y_test, predictions)
mse  = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, predictions)

print(f'\nMAE  : {mae:,.2f}')
print(f'MSE  : {mse:,.2f}')
print(f'RMSE : {rmse:,.2f}')
print(f'R²   : {r2:.4f}')

In [ ]:
# Actual vs Predicted
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predictions, alpha=0.6, edgecolor='k')
lims = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
plt.plot(lims, lims, 'r--', label='perfect prediction')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Linear Regression — Actual vs Predicted')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 🔄 5. Polynomial Regression

Sometimes the relationship is **not linear**. Polynomial Regression adds **powers** of the features:

$$y = b + w_1 x + w_2 x^2 + w_3 x^3$$

### How it works

```python
model = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('regressor', LinearRegression())
])
```

⚠️ **Increasing the degree** makes the model more flexible — but also **more prone to overfitting**.

In [ ]:
# Polynomial regression demo on a curved dataset
np.random.seed(42)
x_demo = np.linspace(-3, 3, 100)
y_demo = 0.5 * x_demo**3 - 2 * x_demo**2 + x_demo + 3 + np.random.randn(100) * 3

X_demo = x_demo.reshape(-1, 1)

# Split
Xtr, Xte, ytr, yte = train_test_split(X_demo, y_demo, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

for ax, deg in zip(axes, [1, 3, 15]):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg)),
        ('reg', LinearRegression()),
    ])
    pipe.fit(Xtr, ytr)
    y_pred_train = pipe.predict(Xtr)
    y_pred_test  = pipe.predict(Xte)
    y_plot = pipe.predict(x_plot)

    ax.scatter(Xtr, ytr, color='red', s=20, label='train', alpha=0.6)
    ax.plot(x_plot, y_plot, 'b-', linewidth=2)
    ax.set_title(f'Degree {deg}  |  train R²={r2_score(ytr, y_pred_train):.3f}  |  test R²={r2_score(yte, y_pred_test):.3f}', fontsize=9)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)

plt.suptitle('Polynomial Regression — degree 1, 3, 15')
plt.tight_layout(); plt.show()

---
## 🛡️ 6. Ridge Regression (L2)

**Ridge** adds an **L2 penalty** to the loss:

$$\text{Loss} = \text{MSE} + \lambda \sum_j w_j^2$$

The penalty **discourages very large weights**.

### When to use Ridge

- Features are correlated
- The model has many features
- You want to reduce overfitting

```python
model = Ridge(alpha=1.0)
```

In [ ]:
# Ridge vs plain Linear Regression
lin = LinearRegression().fit(X_train, y_train)
ridge = Ridge(alpha=1000.0).fit(X_train, y_train)

print(f'{"Feature":<12} {"Linear":>15} {"Ridge":>15}')
print('-' * 44)
for feat, c1, c2 in zip(X.columns, lin.coef_, ridge.coef_):
    print(f'{feat:<12} {c1:>15,.2f} {c2:>15,.2f}')

print(f'\nLinear test R²: {r2_score(y_test, lin.predict(X_test)):.4f}')
print(f'Ridge  test R²: {r2_score(y_test, ridge.predict(X_test)):.4f}')

---
## ✂️ 7. Lasso Regression (L1)

**Lasso** uses **L1 regularization**:

$$\text{Loss} = \text{MSE} + \lambda \sum_j |w_j|$$

Unlike Ridge, Lasso can push some weights **exactly to zero**.

This makes Lasso useful for **feature selection**.

```python
model = Lasso(alpha=0.1)
```

In [ ]:
# Lasso feature selection demo with a large alpha
lasso = Lasso(alpha=100000.0, max_iter=10000).fit(X_train, y_train)

print('Lasso coefficients (with strong alpha):')
for feat, c in zip(X.columns, lasso.coef_):
    status = '  ← zeroed out' if abs(c) < 1e-6 else ''
    print(f'  {feat:10s} → {c:>15,.2f}{status}')

print(f'\nLasso test R²: {r2_score(y_test, lasso.predict(X_test)):.4f}')

---
## ⚖️ 8. ElasticNet

**ElasticNet** combines L1 and L2:

```text
L1 (Lasso) + L2 (Ridge)
```

```python
model = ElasticNet(alpha=0.1, l1_ratio=0.5)
```

`l1_ratio` controls the balance:

```text
l1_ratio = 1.0 → Lasso-like
l1_ratio = 0.0 → Ridge-like
```

In [ ]:
# ElasticNet demo
en = ElasticNet(alpha=100.0, l1_ratio=0.5, max_iter=10000).fit(X_train, y_train)

print('ElasticNet coefficients:')
for feat, c in zip(X.columns, en.coef_):
    print(f'  {feat:10s} → {c:>15,.2f}')

print(f'\nElasticNet test R²: {r2_score(y_test, en.predict(X_test)):.4f}')

---
## 📊 9. Regression Evaluation Metrics

### MAE — Mean Absolute Error

$$\text{MAE} = \frac{1}{n} \sum |y - \hat{y}|$$

**Easy interpretation:** average absolute error.

### MSE — Mean Squared Error

$$\text{MSE} = \frac{1}{n} \sum (y - \hat{y})^2$$

Large errors get **much greater penalties**.

### RMSE — Root Mean Squared Error

$$\text{RMSE} = \sqrt{\text{MSE}}$$

Same **units as the target** — easy to interpret.

### R² — R-Squared

Measures how much of the target's variation is explained by the model.

```text
R² = 1.0  → Perfect fit
R² ≈ 0    → Weak explanatory performance
R² < 0    → Worse than a simple baseline
```

In [ ]:
# Metric sanity check with the README example
actual    = np.array([100, 200, 300])
predicted = np.array([110, 180, 310])

mae  = mean_absolute_error(actual, predicted)
mse  = mean_squared_error(actual, predicted)
rmse = np.sqrt(mse)
r2   = r2_score(actual, predicted)

print('Actual    :', actual.tolist())
print('Predicted :', predicted.tolist())
print('Errors    :', (actual - predicted).tolist())
print(f'\nMAE  : {mae:.2f}')
print(f'MSE  : {mse:.2f}')
print(f'RMSE : {rmse:.2f}')
print(f'R²   : {r2:.4f}')

---
## 📋 10. Regression Model Comparison

| Model | Main Idea | Main Strength |
| --- | --- | --- |
| Linear Regression | Straight-line relationship | Simple baseline |
| Polynomial Regression | Curved relationship | Captures non-linearity |
| Ridge | L2 regularization | Controls large weights |
| Lasso | L1 regularization | Can perform feature selection |
| ElasticNet | L1 + L2 | Balanced regularization |

In [ ]:
# Compare all regressors on the same dataset
models = {
    'Linear':      LinearRegression(),
    'Polynomial':  Pipeline([('poly', PolynomialFeatures(degree=2)),
                             ('reg', LinearRegression())]),
    'Ridge':       Ridge(alpha=1.0),
    'Lasso':       Lasso(alpha=1.0, max_iter=10000),
    'ElasticNet':  ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000),
}

print(f'{"Model":<14} {"Train R²":>10} {"Test R²":>10} {"RMSE":>15}')
print('-' * 52)
for name, m in models.items():
    m.fit(X_train, y_train)
    tr_r2 = r2_score(y_train, m.predict(X_train))
    te_r2 = r2_score(y_test,  m.predict(X_test))
    rmse  = np.sqrt(mean_squared_error(y_test, m.predict(X_test)))
    print(f'{name:<14} {tr_r2:>10.4f} {te_r2:>10.4f} {rmse:>15,.2f}')

---
## ⚠️ 11. Common Regression Problems

### Overfitting

A very complex model performs **great on training data** but **poorly on unseen data**.

**Solutions:** regularization, cross-validation, simpler models, better feature selection.

### Multicollinearity

Features strongly correlated with each other:

```text
House size in sq ft
House size in square meters
```

Both represent almost the same information. This makes coefficient interpretation **unstable** for linear models.

In [ ]:
# Demonstrate multicollinearity
np.random.seed(42)
size_ft = np.random.randint(500, 3000, 200)
size_m2 = size_ft * 0.0929   # perfectly correlated
target  = 5000 * size_ft + np.random.randn(200) * 50000

X_mc = np.column_stack([size_ft, size_m2])
model_mc = LinearRegression().fit(X_mc, target)

print('Coefficients with perfectly correlated features:')
print(f'  size_ft → {model_mc.coef_[0]:>15,.2f}')
print(f'  size_m2 → {model_mc.coef_[1]:>15,.2f}')
print('\nNote: the coefficients are unstable/large — a classic multicollinearity symptom.')

In [ ]:
# Correlation heatmap to detect multicollinearity
plt.figure(figsize=(6, 5))
corr = df.corr(numeric_only=True)
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='correlation')
plt.xticks(range(len(corr)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        plt.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                 color='white' if abs(corr.iloc[i, j]) > 0.5 else 'black', fontsize=8)
plt.title('Feature correlation matrix')
plt.tight_layout(); plt.show()

---
## 📝 12. Practice Checklist

Build a regression model using a **house-price dataset**. Try each of these:

- [ ] Linear Regression
- [ ] Polynomial Regression
- [ ] Ridge
- [ ] Lasso
- [ ] ElasticNet
- [ ] MAE
- [ ] MSE
- [ ] RMSE
- [ ] R²

Then compare their validation/test performance.

---
## 🏋️ 13. Hands-On Exercises

### Exercise 1 — Ridge Alpha Sweep
Try `alpha ∈ {0.01, 0.1, 1, 10, 100, 1000}`. Plot test R² vs alpha. Where does performance peak?

### Exercise 2 — Lasso Feature Selection
Increase `alpha` step by step. At what alpha does the first feature get **zeroed out**? Which is the last to go?

### Exercise 3 — Polynomial Degree Sweep
Fit polynomials of degree `1, 2, 3, 5, 10, 20`. Plot train and test R². Where does overfitting begin?

### Exercise 4 — Metric Interpretation
If RMSE = 250,000 on house prices, what does this mean in practical terms? Is it acceptable?

### Exercise 5 — Multicollinearity Detection
Create two perfectly correlated features and one independent feature. Fit Linear Regression. Which coefficient becomes unstable?

### Exercise 6 — End-to-End
Build a full pipeline with scaling + polynomial + Ridge. Cross-validate with `cross_val_score` and report mean ± std R².

---
## 🎤 14. Interview Questions

1. **What is regression?**  
   Predicting a continuous numerical value from input features.

2. **Regression vs classification?**  
   Regression → continuous output. Classification → discrete class.

3. **Explain Linear Regression.**  
   Fits a linear equation `y = w·x + b` minimizing MSE between predictions and actual values.

4. **What does the slope represent?**  
   How much the target changes when the feature increases by 1 unit.

5. **What is MSE?**  
   Mean Squared Error — average of squared differences between predictions and actual values.

6. **MAE vs MSE?**  
   MAE gives equal weight to all errors; MSE heavily penalizes large errors.

7. **Why use RMSE?**  
   It's in the same units as the target, making it easier to interpret.

8. **What does R² represent?**  
   The fraction of variance in the target explained by the model.

9. **Ridge vs Lasso?**  
   Ridge uses L2 (shrinks weights); Lasso uses L1 (can zero them out → feature selection).

10. **Why does Lasso perform feature selection?**  
    The L1 penalty creates a geometry where some optimal coefficients land exactly at zero.

11. **What is ElasticNet?**  
    A combination of L1 and L2 regularization, controlled by `l1_ratio`.

12. **What is polynomial regression?**  
    Linear regression applied to polynomial (power) features of the input.

13. **What is regularization?**  
    Adding a penalty on weight magnitude to reduce overfitting.

14. **What is multicollinearity?**  
    When features are strongly correlated, making coefficient estimates unstable.

---
## 🏁 15. Key Takeaways

```text
Regression → Predict numerical values

Linear Regression
      ↓
Polynomial Regression
      ↓
Ridge → L2
Lasso → L1
ElasticNet → L1 + L2

MAE  → Average absolute error
MSE  → Penalizes large errors
RMSE → Error in target units
R²   → Explained variance / goodness of fit
```

### 🧭 Golden Rule

> **Choose the metric that matches your problem, and use regularization whenever your model is overfitting. R² alone doesn't tell the whole story — pair it with RMSE.**

---

### 🔗 What's Next?

**Module 04 — Classification**

We move from **predicting numbers** to **predicting categories** — logistic regression, KNN, SVM, Naive Bayes, decision trees, and classification metrics like accuracy, precision, recall, F1, and ROC-AUC.